# IIMT Native: CSV → одна / две руки
Без ROS. `pip install "iimt-native[notebook]"` из проверенного wheel. По умолчанию используются **внешние TCP mock-процессы**, не реальные роботы.

CSV содержит время `t` (секунды) и `left_q1_rad…left_q6_rad`, `right_q1_rad…right_q6_rad`. Скорости/ускорения из CSV не используются и не придумываются. Подвод — отдельная ячейка с RRTConnect. Графики используют **UTC**, ошибка = фактическое − заданное. Timestamp измерения — получение ответа на ПК, не время энкодера. Упреждение изменяет отправку, не ось графика.

На настоящем стенде проверьте URDF/основания/workspace, аварийный останов и допустимость траектории. Mock не доказывает динамику или безопасность железа.

In [ ]:
%matplotlib inline
from pathlib import Path
import time
import numpy as np
from iimt import Arm, Robot, Trajectory, ci05_scene
from iimt.mock import MockRobot
from iimt.plotting import plot_execution

DUAL = True
USE_MOCK = True
VIEWER = False  # True — отдельное 3D окно; нужен графический рабочий стол.
RUN_APPROACH = False
RUN_CSV = False
LEFT_IP, RIGHT_IP = "192.168.50.95", "192.168.50.92"
LEFT_PORT = RIGHT_PORT = 29999
COMMAND_HZ, READ_HZ = 125., 125.
ADVANCE_MS = [0., 0.]  # независимые калиброванные значения, не универсальные 330 мс.
START_DELAY_S = 3.0    # timestamp сообщения создаётся ПОСЛЕ collision preflight.
WORKSPACE_MARGINS = (.5, .5, .5, .5)  # X−, X+, Y−, Y+, метры.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/trajectory_both_TimeGrid_30Hz.csv").exists())
CSV_PATH = ROOT / "data/trajectory_both_TimeGrid_30Hz.csv"
OUTPUT = ROOT / "results" / time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
scene = ci05_scene(dual=DUAL, margins=WORKSPACE_MARGINS)
trajectory = Trajectory.from_csv(CSV_PATH, scene.joint_names, arm="both" if DUAL else "left")
print(len(trajectory.times), "точек; длительность", trajectory.times[-1], "с")

In [ ]:
# Независимые mock-серверы. Контроллер знает только обычные IP и TCP-порты.
mocks = []
endpoints = [(LEFT_IP, LEFT_PORT), (RIGHT_IP, RIGHT_PORT)][:2 if DUAL else 1]
if USE_MOCK:
    for i in range(len(endpoints)):
        mock = MockRobot(positions=trajectory.positions[0, i*6:(i+1)*6].tolist()).start()
        mocks.append(mock)
    endpoints = [(mock.host, mock.port) for mock in mocks]
arms = [Arm(name=("left", "right")[i], ip=ip, port=port,
            joint_names=scene.joint_names[i*6:(i+1)*6], enable_motion=True,
            command_rate=COMMAND_HZ, read_rate=READ_HZ, advance_ms=ADVANCE_MS[i])
        for i, (ip, port) in enumerate(endpoints)]
robot = Robot(arms, scene=scene).connect(viewer=VIEWER)
robot.get_state()

## 1. Отдельный подвод к первой точке CSV через RRTConnect

In [ ]:
approach_execution = None
if RUN_APPROACH:
    try:
        approach = robot.plan(trajectory.positions[0], scale=.2)
        approach_execution = robot.execute_trajectory(approach, start_delay=START_DELAY_S)
        print(approach_execution.wait(approach.times[-1] + START_DELAY_S + 10))
    except Exception as error:
        print(type(error).__name__, error)
    finally:
        if approach_execution is not None:
            figures, metrics = plot_execution(approach_execution, OUTPUT / "approach")
            print(metrics)
else:
    print("Для подвода установите RUN_APPROACH=True и повторите эту ячейку.")

## 2. Выполнить именно CSV
Нет entry/lead-out и переназначения времени. Обе руки получают один `start_unix_ns`. Collision preflight дискретный, не непрерывная гарантия.

In [ ]:
csv_execution = None
if RUN_CSV:
    try:
        csv_execution = robot.execute_trajectory(trajectory, start_delay=START_DELAY_S)
        print(csv_execution.wait(trajectory.times[-1] + START_DELAY_S + 10))
    except Exception as error:
        print(type(error).__name__, error)
    finally:
        if csv_execution is not None:
            figures, metrics = plot_execution(csv_execution, OUTPUT / "csv")
            print(metrics)
        print(robot.diagnostics())
else:
    print("Для выполнения установите RUN_CSV=True и повторите эту ячейку.")

## Дополнительно: случайная суставная цель, RRTConnect и явное выполнение

In [ ]:
RUN_RANDOM = False
random_execution = None
if RUN_RANDOM:
    try:
        measured = np.concatenate([arm["positions"] for arm in robot.get_state()["arms"]])
        target = measured + np.random.default_rng().uniform(-.15, .15, len(measured))
        random_plan = robot.plan(target, scale=.2)  # невалидная цель отвергается.
        random_execution = robot.execute_trajectory(random_plan, start_delay=START_DELAY_S)
        print(random_execution.wait(random_plan.times[-1] + START_DELAY_S + 10))
    except Exception as error:
        print(type(error).__name__, error)
    finally:
        if random_execution is not None:
            plot_execution(random_execution, OUTPUT / "random")

In [ ]:
# robot.cancel() — отдельная явная отмена активного выполнения.
robot.close()
for mock in mocks:
    mock.close()